# Last-Mile Delivery Cost Optimizer — Methodology Notebook

This notebook walks through the full optimization pipeline step-by-step using the Chicago Florist sample scenario.  
It demonstrates the core operations research and data engineering concepts behind the web app.

**Pipeline:**
1. Geocode addresses → lat/lon
2. Build a real driving-distance matrix (OSRM)
3. Solve the Vehicle Routing Problem (Google OR-Tools)
4. Generate a naive sequential baseline
5. Calculate dollar cost savings (BLS wages + EIA fuel prices)
6. Visualize the results


In [ ]:
import sys
from pathlib import Path

# Add src/ to the path so we can import project modules
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print('ROOT:', ROOT)

## 1. Load the sample scenario

The Chicago Florist scenario has 15 delivery stops departing from a Logan Square depot.

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'sample_scenarios' / 'chicago_florist.csv')
print(f'{len(df)} total rows ({len(df) - 1} stops + 1 depot)')
df.head()

In [ ]:
depot_row = df[df['stop_number'] == 0].iloc[0]
stop_rows = df[df['stop_number'] != 0]

all_addresses = [depot_row['address']] + stop_rows['address'].tolist()
print(f'Depot: {depot_row["address"]}')
print(f'Stops ({len(stop_rows)}):')
for addr in all_addresses[1:6]:
    print(f'  {addr}')
print('  ...')

## 2. Geocoding

Each address is resolved to (lat, lon) via the **Nominatim** OpenStreetMap API.  
Rate limit: 1.1 seconds between requests (OpenStreetMap usage policy).

> **Note:** This cell makes real HTTP calls and takes ~18 seconds for 16 addresses.

In [ ]:
from geocoder import geocode_addresses

print('Geocoding addresses (this takes ~18s due to Nominatim rate limits)...')
locations = geocode_addresses(all_addresses)

successes = [l for l in locations if l['success']]
failures  = [l for l in locations if not l['success']]
print(f'\nGeocoded: {len(successes)}/{len(locations)} successfully')
if failures:
    print(f'Failed: {[f["address"] for f in failures]}')

pd.DataFrame(successes)[['address','lat','lon']].head(5)

## 3. Driving Distance Matrix

We call the **OSRM Table API** to get real road distances and travel times for all N×N pairs of locations.  
This is far more accurate than straight-line (Haversine) distances for routing purposes.

In [ ]:
from distance_matrix import build_distance_matrix, meters_to_miles

print('Building distance matrix via OSRM...')
matrix_result = build_distance_matrix(locations)
valid_locations = matrix_result['locations']

n = len(valid_locations)
dist_matrix = matrix_result['distance_matrix']
dur_matrix  = matrix_result['duration_matrix']

print(f'Matrix size: {n}×{n}')
print(f'Fallback used: {matrix_result["fallback"]}')

# Show the distance matrix as a heatmap
dist_miles = [[meters_to_miles(d) for d in row] for row in dist_matrix]
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(dist_miles, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Miles')
ax.set_title(f'Driving Distance Matrix ({n}×{n} locations)', fontsize=13)
ax.set_xlabel('Destination')
ax.set_ylabel('Origin')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
tick_labels = ['Depot'] + [f'S{i}' for i in range(1, n)]
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(tick_labels, fontsize=7)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'distance_matrix_heatmap.png', dpi=120)
plt.show()
print('Saved: outputs/distance_matrix_heatmap.png')

## 4. Vehicle Routing Problem (VRP)

We use **Google OR-Tools** to solve the multi-vehicle routing problem.

### Algorithm
- **Initial solution:** PATH_CHEAPEST_ARC — greedily builds routes by always adding the cheapest next arc
- **Improvement:** GUIDED_LOCAL_SEARCH — metaheuristic that escapes local optima by penalising frequently-used arcs
- **Objective:** minimise total distance across all vehicle routes

### Problem formulation
- N locations (1 depot + N-1 delivery stops)
- K vehicles, each starting and ending at the depot
- Every stop must be visited exactly once
- Minimise: Σ distance(route_k)

In [ ]:
from vrp_solver import solve_vrp, format_routes_readable

NUM_VEHICLES = 2
TIME_LIMIT   = 15  # seconds (adaptive: ~1s per stop)

print(f'Solving VRP: {n-1} stops, {NUM_VEHICLES} vehicles, {TIME_LIMIT}s time limit...')
vrp_result = solve_vrp(
    dist_matrix,
    num_vehicles=NUM_VEHICLES,
    time_limit_seconds=TIME_LIMIT,
    duration_matrix=dur_matrix,
)

print(f'\nStatus: {vrp_result["solver_status"]}')
print(f'Total distance: {meters_to_miles(vrp_result["total_distance_meters"]):.1f} miles')
print(f'Total duration: {vrp_result["total_duration_seconds"]/3600:.2f} hours')
print()
for line in format_routes_readable(vrp_result['routes'], valid_locations):
    print(line)

## 5. Naive Sequential Baseline

The baseline simulates how most small businesses plan routes without software:  
stops are split sequentially across vehicles in the order they appear in the input list.

In [ ]:
from naive_router import solve_naive

naive_result = solve_naive(
    valid_locations,
    num_vehicles=NUM_VEHICLES,
    distance_matrix=dist_matrix,
    duration_matrix=dur_matrix,
)

print(f'Status: {naive_result["solver_status"]}')
print(f'Total distance: {meters_to_miles(naive_result["total_distance_meters"]):.1f} miles')
print(f'Total duration: {naive_result["total_duration_seconds"]/3600:.2f} hours')
print()
for line in format_routes_readable(naive_result['routes'], valid_locations):
    print(line)

## 6. Cost Savings Analysis

Costs are calculated using real-world data:
- **Driver wage:** BLS OES SOC 53-3031 (Light Truck Drivers), Chicago-Naperville-Elgin metro
- **Fuel price:** EIA 2024 Illinois state average retail price

In [ ]:
from cost_calculator import calculate_savings_report, get_driver_wage, get_fuel_price

wage       = get_driver_wage('Chicago', 'IL')
fuel_price = get_fuel_price('IL', 'gasoline')
print(f'Chicago driver wage (BLS SOC 53-3031): ${wage:.2f}/hr')
print(f'Illinois gasoline price (EIA 2024):    ${fuel_price:.2f}/gal')

report = calculate_savings_report(
    naive_result=naive_result,
    optimized_result=vrp_result,
    city='Chicago',
    state='IL',
    num_stops=n - 1,
    vehicle_type='van',
    fuel_type='gasoline',
)

print()
print('=== SAVINGS REPORT ===')
print(f'Naive cost:             ${report["naive_total_cost"]:.2f}')
print(f'Optimized cost:         ${report["optimized_total_cost"]:.2f}')
print(f'Daily savings:          ${report["savings_usd"]:.2f}  ({report["savings_pct"]}%)')
print(f'Annual savings (250d):  ${report["annual_savings_usd"]:,.2f}')
print(f'Miles saved:            {report["distance_saved_miles"]:.1f} mi')
print(f'Time saved:             {report["time_saved_hours"]*60:.0f} min')
print(f'CO\u2082 avoided:           {report["co2_saved_lbs"]:.1f} lbs')

In [ ]:
# Cost breakdown bar chart
bd = report['breakdown']
categories = ['Naive', 'Optimized']
driver_costs = [bd['naive_driver_cost'], bd['optimized_driver_cost']]
fuel_costs   = [bd['naive_fuel_cost'],   bd['optimized_fuel_cost']]

x = np.arange(len(categories))
width = 0.45

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Stacked bar chart
ax = axes[0]
bars1 = ax.bar(x, driver_costs, width, label='Driver cost', color='#6C7EF8')
bars2 = ax.bar(x, fuel_costs,   width, bottom=driver_costs, label='Fuel cost', color='#3AB8A0')
ax.set_ylabel('Cost (USD)')
ax.set_title('Cost Breakdown: Naive vs Optimized')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
totals = [d + f for d, f in zip(driver_costs, fuel_costs)]
for i, (bar, total) in enumerate(zip(bars1, totals)):
    ax.text(bar.get_x() + bar.get_width()/2, total + 0.2, f'${total:.2f}',
            ha='center', va='bottom', fontweight='bold', fontsize=10)

# Distance comparison
ax2 = axes[1]
dist_vals = [report['naive_distance_miles'], report['optimized_distance_miles']]
colors = ['#E05C3A', '#3AB8A0']
bars = ax2.bar(categories, dist_vals, color=colors, width=0.45)
ax2.set_ylabel('Miles')
ax2.set_title('Total Distance: Naive vs Optimized')
for bar, val in zip(bars, dist_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.2, f'{val:.1f} mi',
             ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.suptitle(
    f'Chicago Florist — {n-1} stops, {NUM_VEHICLES} vehicles\n'
    f'Daily saving: ${report["savings_usd"]:.2f} ({report["savings_pct"]}%)  |  '
    f'Annual: ${report["annual_savings_usd"]:,.0f}',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'cost_savings_chart.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/cost_savings_chart.png')

## 7. Sensitivity Analysis

How do annual savings change as the number of delivery stops increases?

In [ ]:
# Show how savings % varies with number of vehicles
# Use the actual report values from our Chicago run

# Illustrative sensitivity: savings as a function of stop count
# (using a simplified model: savings_pct is roughly constant, cost scales with stops)
stop_counts = list(range(5, 26, 5))
base_savings_pct = report['savings_pct'] / 100

# Estimate: cost scales roughly linearly with stops
cost_per_stop = report['naive_total_cost'] / (n - 1)
annual_savings_by_stop = [
    round(cost_per_stop * s * base_savings_pct * 250, 0)
    for s in stop_counts
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(stop_counts, annual_savings_by_stop, color='#6C7EF8', width=3.5)
ax.set_xlabel('Number of daily delivery stops')
ax.set_ylabel('Projected annual savings (USD)')
ax.set_title('Estimated Annual Savings by Route Size\n(Chicago Florist — van @ 18 mpg)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, (s, val) in enumerate(zip(stop_counts, annual_savings_by_stop)):
    ax.text(s, val + 50, f'${val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'sensitivity_analysis.png', dpi=120)
plt.show()
print('Saved: outputs/sensitivity_analysis.png')

## 8. Key takeaways

| Metric | Value |
|---|---|
| Algorithm | OR-Tools GUIDED_LOCAL_SEARCH |
| Distance source | OSRM real road distances |
| Wage source | BLS OES SOC 53-3031, metro level |
| Fuel source | EIA 2024 state average |
| Time to solve | Adaptive (≈1s/stop, max 55s) |

### Why not TSP?
The Travelling Salesman Problem (TSP) finds the single shortest tour through all nodes.  
The **VRP** is a generalization: multiple vehicles, each starting and ending at the depot.  
OR-Tools handles VRP natively via its `RoutingModel` — TSP is just VRP with K=1.

### Why GUIDED_LOCAL_SEARCH?
GLS penalises arcs that appear frequently in locally-optimal solutions, pushing the solver  
to explore different regions of the solution space. It consistently outperforms pure local  
search on real-world VRP instances with 10–100 stops.

### Limitations
- No time-window constraints (all stops treated as available any time)
- No vehicle capacity constraints (package weight is not optimized)
- OSRM uses pre-computed road graphs — live traffic not included
- BLS/EIA data is annual averages — actual costs vary by time of day and season